# 12 - pixel size or radar band? (experiment B, validation glacier SI only)

**A100 GPU runtime** (standard RAM is enough: notebook 11 used 12 GB). Measured: 9.6 min.
Nothing is retrained and no result file is changed; the test glaciers are not touched.

In CaFFe, X-band images are always 7 m pixels and C-band mostly 20 m, so the per-band
results cannot tell band from pixel size. Here the 32 X-band (TerraSAR-X, 7 m) images of
SI are shrunk to 20 m and the SAVED models are run on them unchanged: U-Net, C-RADIO and
satellite-DINOv3 probes (row 1) and decoders (row 3, seeds 0/1/2).

1. **Control:** the heads on the original 7 m images must reproduce the saved SI results
   exactly, or the run stops. The U-Net's saved SI file came from the model in memory
   right after training, so its X@7, X@20 and C@20 are all rerun on one path and its
   comparison with the old file is shown for information.
2. **Result:** pooled front error of X-band at 7 m, X-band shrunk to 20 m, and the SI
   C-band images that really are 20 m.

**Pre-registered rule** (fixed on 2026-09-27 before the run; reproduced in docs/results.md):
r = (X@20 - X@7) / (C@20 - X@7). r >= 0.5 -> pixel size explains the X-vs-C difference for
that model; r < 0.5 -> it does not; |C@20 - X@7| < 100 m -> nothing to explain.
Caveats: at equal pixel size the rest still mixes band with satellite, year and scene;
a shrunk image is not a real 20 m product; 32 images; exploratory.

In [1]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# --- cell: secrets, GPU, RAM, data ---
import os, shutil
from pathlib import Path
import psutil, torch
from sartransfer.env import load_env, require, model_cache_on_drive
from sartransfer.data import inventory as inv

load_env()
require("HF_TOKEN", "gated model weights")
model_cache_on_drive()          # weights downloaded once, kept on Drive
assert torch.cuda.is_available(), "switch to a GPU runtime"
g = torch.cuda.get_device_properties(0)
assert "A100" in g.name, f"got {g.name}: use an A100 runtime -- the saved SI results were made on an A100, and the exact control needs the same GPU kernels"
print(f"gpu: {g.name} | GPU memory {g.total_memory / 1e9:.0f} GB | system RAM {psutil.virtual_memory().total / 1e9:.0f} GB "
      f"| CPU cores {os.cpu_count()}")
DRIVE = Path("/content/drive/MyDrive")
RAW, RES = DRIVE / "sar-transfer" / "raw", DRIVE / "sar-transfer" / "results"
inv.unzip_caffe(inv.fetch_caffe_zip(RAW), Path("/content/caffe"))
DATA = inv.find_data_root(Path("/content/caffe"))
df = inv.scan(DATA, subdirs=("sar_images",))
WORK = Path("/content/restest")
print(len(df), "SAR images | free disk:", f"{shutil.disk_usage('/content').free/1e9:.0f} GB")

In [4]:
# --- cell: RUN -- shrink, control at 7 m, run at 20 m, table (~10 min) ---
# The saved output below was printed before the columns were renamed on 2026-09-27:
# X_7m = native, X_20m = shrunk_20m, no_front_7m = no_front_native. Same numbers.
# A rerun now writes to /content/restest/X7, as restest_X7_*.csv; the next cell's saved output
# shows the old file names (restest_table.csv, restest_records_7m.csv, ...).
import pandas as pd
from sartransfer.resolution import run_resolution_test

control, table = run_resolution_test(df, DATA, RES, WORK, token=os.environ["HF_TOKEN"])
pd.set_option("display.width", 250)
print(table[["model", "native", "shrunk_20m", "C_20m", "r", "verdict", "median_change_per_image_m",
             "no_front_native", "no_front_20m"]].round({"native": 0, "shrunk_20m": 0, "C_20m": 0, "r": 2,
                                                       "median_change_per_image_m": 0}).to_string(index=False))

shrunk 32 SI TSX images to 20 m pixels -> /content/restest

--- 7 m (original, control): 32 X-band images ---
installing ['open_clip_torch']


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

remote code verified: 26 files == reviewed copy (nvidia/C-RADIOv4-H@0057b339059c)


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

...

remote code verified: 26 files == reviewed copy (nvidia/C-RADIOv4-H@0057b339059c)


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

cradio-v4-h: 651.6 M parameters loaded
restricted to splits ('val',): 32 of 32 images
  32/32  0.0 min elapsed, 0.08 s/image, ETA 0.0 min
in memory: train X (0, 0) (0.0 GB fp16), 32 val images
extraction time: waiting for CPU preparation 0.0 min, encoder 0.0 min, keeping/copying 0.0 min, total 0.0 min


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
restricted to splits ('val',): 32 of 32 images
  32/32  0.0 min elapsed, 0.06 s/image, ETA 0.0 min
in memory: train X (0, 0) (0.0 GB fp16), 32 val images
extraction time: waiting for CPU preparation 0.0 min, encoder 0.0 min, keeping/copying 0.0 min, total 0.0 min

done in 9.6 min
          model   X_7m  X_20m  C_20m     r                                   verdict  median_change_per_image_m  no_front_7m  no_front_20m
    cradio_row1 1821.0 2497.0 2060.0  2.83         pixel size explains it (r >= 0.5)                      401.0            0             0
 cradio_row3_s0 1158.0 1710.0 1497.0  1.63         pixel size explains it (r >= 0.5)                      620.0            0             0
 cradio_row3_s1 1092.0 1289.0 1302.0  0.94         pixel size explains it (r >= 0.5)                      228.0            0             0
 cradio_row3_s2 1233.0 1373.0 1409.0  0.79         pixel size explains it (r >= 0.5)                      166.0            

In [5]:
# --- cell: save the result tables to Drive (small CSVs) ---
dst = DRIVE / "sar-transfer" / "resolution_test"
dst.mkdir(parents=True, exist_ok=True)
for f in sorted(WORK.rglob("restest_*.csv")):
    shutil.copy2(f, dst / f.name)
print("saved:", sorted(p.name for p in dst.glob("restest_*.csv")))

saved: ['restest_control.csv', 'restest_images.csv', 'restest_records_20m.csv', 'restest_records_7m.csv', 'restest_table.csv']
